# 03 — Feature Engineering

Loads all raw team game log CSVs, applies every feature function from `src/features.py`, and saves the result to `data/processed/features.csv`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import glob

import pandas as pd

from src.features import (
    add_home_away_flag,
    add_last_10_wins,
    add_rest_days,
    add_rolling_averages,
    add_win_streak,
)

RAW_DIR = Path.cwd().parent / "data" / "raw"
PROCESSED_DIR = Path.cwd().parent / "data" / "processed"

# stat columns to compute rolling averages for
STAT_COLS = ["PTS", "FG_PCT", "FG3_PCT", "FT_PCT", "REB", "AST", "TOV", "STL", "BLK"]

## 1. Load Raw Team Game Logs

In [ ]:
csv_paths = sorted(RAW_DIR.glob("team_game_logs_*.csv"))
print(f"{len(csv_paths)} team-season files found")

raw_df = pd.concat([pd.read_csv(p) for p in csv_paths], ignore_index=True)
print(f"Combined shape: {raw_df.shape}")
raw_df.head(3)

## 2. Apply Features Per Team

Each feature function sorts by date internally, so we loop over individual team-season files rather than applying across the whole DataFrame at once. This keeps win streaks, rest days, and rolling averages scoped to each team's own game history.

In [ ]:
enriched = []

for p in csv_paths:
    df = pd.read_csv(p)

    df = add_home_away_flag(df)
    df = add_rest_days(df)
    df = add_rolling_averages(df, stat_cols=STAT_COLS, window=5)
    df = add_rolling_averages(df, stat_cols=STAT_COLS, window=10)
    df = add_win_streak(df)
    df = add_last_10_wins(df)

    enriched.append(df)

features_df = pd.concat(enriched, ignore_index=True)
print(f"Feature matrix shape: {features_df.shape}")
features_df.head(3)

## 3. Encode Target Variable

In [ ]:
features_df["target"] = (features_df["WL"] == "W").astype(int)

print("Class balance:")
print(features_df["target"].value_counts(normalize=True).rename({1: "Win", 0: "Loss"}))

## 4. Inspect New Columns

In [ ]:
feature_cols = (
    ["is_home", "rest_days", "win_streak", "last_10_w"]
    + [f"{c}_roll5"  for c in STAT_COLS]
    + [f"{c}_roll10" for c in STAT_COLS]
)

print(f"{len(feature_cols)} feature columns:")
print(feature_cols)

features_df[["GAME_DATE", "MATCHUP", "WL"] + feature_cols + ["target"]].head(10)

In [ ]:
null_counts = features_df[feature_cols].isnull().sum()
print("Null counts per feature column:")
print(null_counts[null_counts > 0])

## 5. Save to Processed

In [ ]:
out_path = PROCESSED_DIR / "features.csv"
features_df.to_csv(out_path, index=False)
print(f"Saved {len(features_df):,} rows × {len(features_df.columns)} columns → {out_path}")